# Chapter 7: Data Quality Gates

This notebook demonstrates the three-tier defense for data quality:

1. **Tier 1: Schema Validation** (Pandera)
2. **Tier 2: Business Rules** (DuckDB)
3. **Tier 3: Pipeline Contracts** (Pre/post conditions)

## Setup

In [1]:
import polars as pl
# In Pandera 0.20+ the polars integration lives at `pandera.polars`, and that
# single import exposes DataFrameSchema, Column, Check, and errors.
import pandera as _pandera  # only used here for the version string
import pandera.polars as pa
import duckdb
from datetime import date, datetime, timedelta
import random

print(f"Polars version: {pl.__version__}")
print(f"Pandera version: {_pandera.__version__}")
print(f"DuckDB version: {duckdb.__version__}")

Polars version: 1.41.0
Pandera version: 0.31.1
DuckDB version: 1.5.3


## Generate Sample Data

Let's create a sample orders dataset with some intentional quality issues:

In [2]:
random.seed(42)

# Good orders — note we use `date(...)` so the column lands as polars Date,
# which is what the schema below expects.
good_orders = [
    {"order_id": f"O{i:03d}", "customer_id": random.randint(1, 100),
     "order_date": date(2024, 10, 1) + timedelta(days=random.randint(0, 30)),
     "amount": round(random.uniform(10, 500), 2),
     "quantity": random.randint(1, 10),
     "status": random.choice(["pending", "shipped", "delivered"])}
    for i in range(1, 96)
]

# Add quality issues
bad_orders = [
    {"order_id": "O096", "customer_id": 1, "order_date": date(2024, 10, 15),
     "amount": -99999.0, "quantity": 1, "status": "pending"},  # Sentinel value
    {"order_id": "O097", "customer_id": 2, "order_date": date(2024, 10, 16),
     "amount": 50.0, "quantity": 0, "status": "shipped"},  # Zero quantity
    {"order_id": "O098", "customer_id": 3, "order_date": date(2025, 1, 1),
     "amount": 100.0, "quantity": 5, "status": "delivered"},  # Future date
    {"order_id": "O099", "customer_id": 4, "order_date": date(2024, 10, 20),
     "amount": 200.0, "quantity": 2, "status": "shippd"},  # Typo in status
    {"order_id": "O099", "customer_id": 5, "order_date": date(2024, 10, 21),
     "amount": 300.0, "quantity": 3, "status": "pending"},  # Duplicate ID
]

all_orders = good_orders + bad_orders
# Cast quantity to Int32 to match the schema dtype below.
df = pl.DataFrame(all_orders).with_columns(pl.col("quantity").cast(pl.Int32))

print(f"Generated {len(df)} orders with {len(bad_orders)} quality issues")
df.tail(10)

Generated 100 orders with 5 quality issues


order_id,customer_id,order_date,amount,quantity,status
str,i64,date,f64,i32,str
"""O091""",43,2024-10-26,466.1,4,"""shipped"""
"""O092""",21,2024-10-26,353.63,7,"""pending"""
"""O093""",61,2024-10-08,107.79,8,"""shipped"""
"""O094""",40,2024-10-27,399.72,4,"""pending"""
"""O095""",4,2024-10-22,104.63,6,"""shipped"""
"""O096""",1,2024-10-15,-99999.0,1,"""pending"""
"""O097""",2,2024-10-16,50.0,0,"""shipped"""
"""O098""",3,2025-01-01,100.0,5,"""delivered"""
"""O099""",4,2024-10-20,200.0,2,"""shippd"""


## Tier 1: Schema Validation with Pandera

Define schema with type checks, null constraints, and value ranges:

In [3]:
orders_schema = pa.DataFrameSchema({
    "order_id": pa.Column(pl.Utf8, unique=True, nullable=False),
    "customer_id": pa.Column(pl.Int64, nullable=False),
    "order_date": pa.Column(pl.Date, nullable=False),
    "amount": pa.Column(
        pl.Float64,
        checks=[
            pa.Check.greater_than(0),
            pa.Check.less_than(1_000_000)
        ]
    ),
    "quantity": pa.Column(
        pl.Int32,
        checks=[
            pa.Check.greater_than_or_equal_to(1),
            pa.Check.less_than_or_equal_to(100)
        ]
    ),
    "status": pa.Column(
        pl.Utf8,
        checks=pa.Check.isin(["pending", "shipped", "delivered", "cancelled"])
    ),
})

print("Schema defined with checks for:")
print("- Unique order_id")
print("- Amount > 0 and < 1M")
print("- Quantity between 1-100")
print("- Status in allowed values")

Schema defined with checks for:
- Unique order_id
- Amount > 0 and < 1M
- Quantity between 1-100
- Status in allowed values


### Run Schema Validation

In [4]:
try:
    validated_df = orders_schema.validate(df, lazy=True)
    print(f"[OK] Validated {len(validated_df):,} rows")
except pa.errors.SchemaErrors as e:
    print("[FAIL] Validation failed:\n")
    failure_cases = e.failure_cases
    print(failure_cases)
    print(f"\nFound {len(failure_cases)} validation errors")

[FAIL] Validation failed:

shape: (5, 6)
┌──────────────┬────────────────┬──────────┬─────────────────────────────┬──────────────┬───────┐
│ failure_case ┆ schema_context ┆ column   ┆ check                       ┆ check_number ┆ index │
│ ---          ┆ ---            ┆ ---      ┆ ---                         ┆ ---          ┆ ---   │
│ str          ┆ str            ┆ str      ┆ str                         ┆ i32          ┆ i32   │
╞══════════════╪════════════════╪══════════╪═════════════════════════════╪══════════════╪═══════╡
│ O099         ┆ Column         ┆ order_id ┆ field_uniqueness            ┆ null         ┆ 98    │
│ O099         ┆ Column         ┆ order_id ┆ field_uniqueness            ┆ null         ┆ 99    │
│ -99999.0     ┆ Column         ┆ amount   ┆ greater_than(0)             ┆ 0            ┆ 95    │
│ 0            ┆ Column         ┆ quantity ┆ greater_than_or_equal_to(1) ┆ 0            ┆ 96    │
│ shippd       ┆ Column         ┆ status   ┆ isin(['pending', 'shipped', ┆ 0 

## Tier 2: Business Rules with DuckDB

Go beyond schema - validate business logic:

In [5]:
# Clean the data for this demo
clean_df = df.filter(
    (pl.col("amount") > 0) &
    (pl.col("quantity") >= 1) &
    (pl.col("status").is_in(["pending", "shipped", "delivered"]))
).unique(subset=["order_id"])

print(f"Cleaned to {len(clean_df)} rows for business rules demo")

Cleaned to 97 rows for business rules demo


In [6]:
# Initialize DuckDB
con = duckdb.connect(":memory:")

# Create quality log table
con.execute("""
    CREATE TABLE quality_log (
        check_name VARCHAR,
        check_type VARCHAR,
        failing_rows BIGINT,
        check_query TEXT,
        run_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

# Load orders
con.execute("CREATE TABLE orders AS SELECT * FROM clean_df")

print("[OK] DuckDB initialized with orders table")

[OK] DuckDB initialized with orders table


### Check 1: Statistical Outlier Detection

In [7]:
# Find revenue outliers (>3 std devs from mean)
outliers = con.execute("""
    WITH stats AS (
        SELECT
            AVG(amount) as mean,
            STDDEV(amount) as std
        FROM orders
    )
    SELECT
        order_id,
        amount,
        (amount - stats.mean) / stats.std as z_score
    FROM orders, stats
    WHERE ABS(amount - stats.mean) > 3 * stats.std
""").pl()

if len(outliers) > 0:
    print("[WARNING] Found statistical outliers:")
    print(outliers)
else:
    print("[OK] No statistical outliers detected")

[OK] No statistical outliers detected


### Check 2: Temporal Logic

In [8]:
# Check for future dates
future_orders = con.execute("""
    SELECT order_id, order_date
    FROM orders
    WHERE order_date > CURRENT_DATE
""").pl()

if len(future_orders) > 0:
    print("[FAIL] Found orders with future dates:")
    print(future_orders)
else:
    print("[OK] No future-dated orders")

[OK] No future-dated orders


## Tier 3: Pipeline Contracts

Ensure transformations work correctly with pre/post conditions:

In [9]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class DataContract:
    name: str
    pre_conditions: list[Callable]
    post_conditions: list[Callable]

def run_with_contract(contract, transform, df):
    # Pre-conditions
    for i, check in enumerate(contract.pre_conditions):
        if not check(df):
            raise ValueError(f"{contract.name}: pre-condition {i} failed")

    # Transform
    result = transform(df)

    # Post-conditions
    for i, check in enumerate(contract.post_conditions):
        if not check(result):
            raise ValueError(f"{contract.name}: post-condition {i} failed")

    return result

print("[OK] Contract framework ready")

[OK] Contract framework ready


In [10]:
# Define contract for daily aggregation
aggregate_contract = DataContract(
    name="daily_revenue_aggregation",
    pre_conditions=[
        lambda df: len(df) > 0,
        lambda df: df["order_id"].null_count() == 0,
    ],
    post_conditions=[
        lambda df: len(df) > 0,
        lambda df: (df["revenue"] >= 0).all(),
    ]
)

def aggregate_daily_revenue(df: pl.DataFrame) -> pl.DataFrame:
    return df.group_by("order_date").agg([
        pl.col("amount").sum().alias("revenue"),
        pl.col("order_id").count().alias("order_count"),
    ])

# Run with contract
try:
    daily_revenue = run_with_contract(
        aggregate_contract,
        aggregate_daily_revenue,
        clean_df
    )
    print("[OK] Contract validation passed")
    print(f"\nDaily revenue summary:")
    print(daily_revenue.sort("order_date").head(10))
except ValueError as e:
    print(f"[FAIL] {e}")

[OK] Contract validation passed

Daily revenue summary:
shape: (10, 3)
┌────────────┬─────────┬─────────────┐
│ order_date ┆ revenue ┆ order_count │
│ ---        ┆ ---     ┆ ---         │
│ date       ┆ f64     ┆ u32         │
╞════════════╪═════════╪═════════════╡
│ 2024-10-01 ┆ 343.06  ┆ 2           │
│ 2024-10-02 ┆ 348.07  ┆ 2           │
│ 2024-10-03 ┆ 795.28  ┆ 3           │
│ 2024-10-04 ┆ 1272.72 ┆ 7           │
│ 2024-10-05 ┆ 1836.52 ┆ 6           │
│ 2024-10-06 ┆ 694.75  ┆ 3           │
│ 2024-10-07 ┆ 1030.61 ┆ 3           │
│ 2024-10-08 ┆ 1178.99 ┆ 5           │
│ 2024-10-10 ┆ 1027.35 ┆ 4           │
│ 2024-10-11 ┆ 60.08   ┆ 1           │
└────────────┴─────────┴─────────────┘


## Summary

### What We Caught

**Tier 1 (Schema):**
- Sentinel value (-99999)
- Zero quantity
- Invalid status (typo)
- Duplicate order_id

**Tier 2 (Business Rules):**
- Statistical outliers
- Future dates
- Referential integrity (if customers table existed)

**Tier 3 (Contracts):**
- Pre: Non-empty input, no null IDs
- Post: Non-negative revenue, preserved totals

### Cost Comparison

| Approach | Runtime | Cost |
|----------|---------|------|
| Local (this) | <1s | $0 |
| Cloud SaaS | Similar | $200/mo |

### Key Takeaways

1. **Three tiers catch different issues** - Use all three
2. **Fail fast on critical** - Schema and contracts
3. **Log warnings for investigation** - Outliers, orphans
4. **Local quality is free** - No cloud bills